In [ ]:
import os
import glob
import cv2
import shutil
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

%matplotlib inline

In [ ]:
import random
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder

In [ ]:
seed = 42
np.random.seed(seed)
random.seed(seed)

In [ ]:
file_path = "./data/raw/01.xml"
tree = ET.parse(file_path)
root = tree.getroot()

In [ ]:
directory = './data/raw/'
label_directory = './data/labels/'

In [ ]:
image_files = glob.glob(os.path.join(directory, '*.jpg')) + glob.glob(os.path.join(directory, '*.png'))\
    + glob.glob(os.path.join(directory, '*.JPG')) + glob.glob(os.path.join(directory, '*.PNG'))

xml_files = glob.glob(os.path.join(directory, '*.xml'))

In [ ]:
len(xml_files), len(image_files)

In [ ]:
info = {
    'image_name':[],
    'width':[],
    'height':[],
    'vehicle':[],
    'xmin':[],
    'ymin':[],
    'xmax':[],
    'ymax':[]
}

In [ ]:
for img in image_files:
    file_name = img.split('/')[-1]
    xml = file_name.split('.')[0]+'.xml'
    with open(os.path.join(directory, xml), 'r') as xml_file:
        root = ET.parse(xml_file).getroot()
        width, height = int(root[4][0].text), int(root[4][1].text)
        for obj in root.iter('object'):
            vehicle_name = obj[0].text
            xmin, ymin, xmax, ymax = int(obj[4][0].text), int(obj[4][1].text), int(obj[4][2].text), int(obj[4][3].text)
            info['image_name'].append(file_name)
            info['width'].append(width)
            info['height'].append(height)
            info['vehicle'].append(vehicle_name)
            info['xmin'].append(xmin)
            info['ymin'].append(ymin)
            info['xmax'].append(xmax)
            info['ymax'].append(ymax)

In [ ]:
data = pd.DataFrame(info)
data.head(50)

In [ ]:
data['image_name'].value_counts()

In [ ]:
data['vehicle'].value_counts()

In [ ]:
val = data[(data['width']== 0) & (data['height'] == 0)]['image_name'].values

In [ ]:
width, height = [], []
for img in val:
    f_name = os.path.join(directory, img)
    image = cv2.imread(f_name)
    h, w, _ = image.shape
    width.append(w)
    height.append(h)

In [ ]:
filt = data[(data['width']== 0) & (data['height'] == 0)].index
data.loc[filt, 'width'] = width
data.loc[filt, 'height'] = height

In [ ]:
data.tail(30)

In [ ]:
plt.figure(figsize=(10, 10))
sns.countplot(data=data, y='vehicle')

In [ ]:
le = LabelEncoder()
le.fit(data['vehicle'])

In [ ]:
le.classes_

In [ ]:
data['label'] = le.transform(data['vehicle'])
data

In [ ]:
x_center = (data['xmax'] + data['xmin'])/2
x_center_normalized = x_center/data['width']
y_center = (data['ymax'] + data['ymin'])/2
y_center_normalized = y_center/data['height']

In [ ]:
x_width = (data['xmax'] - data['xmin'])/data['width']
y_width = (data['ymax'] - data['ymin'])/data['height']

In [ ]:
data['x_center'] = x_center_normalized
data['y_center'] = y_center_normalized
data['x_width'] = x_width
data['y_height'] = y_width

In [ ]:
data

In [ ]:
label_to_idx = {value:index for index, value in enumerate(le.classes_)}
idx_to_label = {value:key for key, value in label_to_idx.items()}
print(label_to_idx)
print(idx_to_label)

In [ ]:
def show_image_annotation(index, data):
    image_name = data.loc[index, 'image_name']
    xml = image_name.split('.')[0]+'.xml'
    root = ET.parse(os.path.join(directory, xml)).getroot()
    image = cv2.imread(os.path.join(directory, image_name))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    for obj in root.iter('object'):
        text = obj[0].text
        xmin, ymin, xmax, ymax = int(obj[4][0].text), int(obj[4][1].text), int(obj[4][2].text), int(obj[4][3].text)
        image = cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (255, 0, 0), 1)
        image = cv2.putText(image, text, org=(xmin, ymin), fontScale=1, color=(255, 0, 0), thickness=1, fontFace=cv2.FONT_HERSHEY_SIMPLEX)
    
    plt.figure(figsize=(15, 15))
    plt.imshow(image)

In [ ]:
show_image_annotation(1000, data)

<h2> Convert XML to YOLOV8 Format </h2>

In [ ]:
# label_directory = "F:/Compressed/labels/"
# group_by_image = data.groupby('image_name')
# for key, grp in group_by_image:
#     file_name = key.split('.')[0]+'.txt'
#     file_name = os.path.join(label_directory, file_name)
    
#     lines = []
#     for row in grp.iterrows():
#         values = row[1][['label', 'x_center', 'y_center', 'x_width', 'y_height']].values
#         lines.append(" ".join([str(val) for val in values]))
    
#     with open(file_name, 'w') as file:
#         for line in lines:
#             file.write(line)
#             file.write('\n')

<h2> Make Cross-Validation Fold </h2>

In [ ]:
folds = 5
# skfold = StratifiedKFold(n_splits=folds, random_state=seed)
sgkf = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)

In [ ]:
data['fold'] = -1
for i, (train_idx, test_idx) in enumerate(sgkf.split(data, data['label'] ,data['image_name'])):
    data.loc[test_idx, 'fold'] = i

In [ ]:
plt.figure(figsize=(10, 10))
sns.countplot(data, y='fold', hue='vehicle')

In [ ]:
# dist = data.groupby('fold')['label'].value_counts()
# data[(data['label'] == 6)]['fold'].value_counts()

<h2> Create Data Folder </h2>

In [ ]:
groupby_fold = data.groupby(by='fold')
for key, grp in groupby_fold:
    
    os.makedirs(f"fold_{key}/images")
    os.makedirs(f"fold_{key}/labels")
    
    unique_images = grp['image_name'].unique()
    
    for image in unique_images:
        img_name = image
        label_name = img_name.split('.')[0]+'.txt'
        
        img_src = directory + img_name
        img_dst = f"fold_{key}/images/" + img_name
        
        label_src = label_directory + label_name
        label_dst = f"fold_{key}/labels/" + label_name
        
        os.symlink(img_src, img_dst)
        os.symlink(label_src, label_dst)

# shutil.rmtree('./fold_0')

<h2> Write yaml File </h2>

In [ ]:
%%writefile object_detection.yaml
path: ./
train:
  - fold_0/
  - fold_1/
  - fold_2/
  - fold_3/
val: fold_4/

names:
  0: ambulance
  1: army vehicle
  2: auto rickshaw
  3: bicycle
  4: bus
  5: car
  6: garbagevan
  7: human hauler
  8: minibus
  9: minivan
  10: motorbike
  11: pickup
  12: policecar
  13: rickshaw
  14: scooter
  15: suv
  16: taxi
  17: three wheelers (CNG)
  18: truck
  19: van
  20: wheelbarrow


In [ ]:
!pip install ultralytics -q

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8m.pt')

In [ ]:
model

In [ ]:
# Historical tracking cell removed during sanitization.
# Configure external experiment tracking through environment variables if reintroduced.


In [ ]:
results = model.train(

batch=16,
device= device,
data="./object_detection.yaml",
save=True,
save_period=20,
epochs=100,
imgsz=800,
dropout=0.0,
# pretrained=True,
    
lr0= 0.01,
lrf= 0.2,
momentum= 0.937,
weight_decay= 0.0005,
warmup_epochs= 3.0,
warmup_momentum= 0.8,
warmup_bias_lr= 0.1,
box= 0.05,
hsv_h= 0.015,
hsv_s= 0.7,
hsv_v= 0.4,
degrees= 0.0,
translate= 0.1,
scale= 0.5,
shear= 0.0,
perspective= 0.0,
flipud= 0.0,
fliplr= 0.5,
mosaic= 1.0,
mixup= 0.0
)

In [ ]:
# results = model.train( 
#         batch=16,
#         device= device,
#         data="./object_detection.yaml",
#         save=True,
#         save_period=20,
#         epochs=100,
#         imgsz=800,
#         dropout=0.0,
#         pretrained=True,
    
# #         augmentation
#         translate=0.1,
#         scale=0.3,
#         fliplr=0.3,
#         mosaic=0.1,
#         shear=0.2,
    
# #         optimizer
#         lr0=0.001,
#         cos_lr = False,
# #         patience=3,
#         optimizer="Adam",
#         momentum=0.947,
#         weight_decay=0.0005
#     )

In [ ]:
wandb.finish()

<h2> Inference </h2>

In [ ]:
# Hardcoded RGB color tuples for 21 keys (0-20)
color_dict = {
    0: (255, 0, 0),   # Red
    1: (0, 255, 0),   # Green
    2: (0, 0, 255),   # Blue
    3: (255, 255, 0), # Yellow
    4: (255, 0, 255), # Magenta
    5: (0, 255, 255), # Cyan
    6: (128, 0, 0),   # Maroon
    7: (0, 128, 0),   # Dark Green
    8: (0, 0, 128),   # Navy
    9: (128, 128, 0), # Olive
    10: (128, 0, 128),# Purple
    11: (0, 128, 128),# Teal
    12: (192, 192, 192), # Silver
    13: (128, 128, 128), # Gray
    14: (255, 165, 0),   # Orange
    15: (0, 128, 255),   # Sky Blue
    16: (255, 0, 128),   # Pink
    17: (255, 128, 0),   # Coral
    18: (128, 0, 255),   # Violet
    19: (0, 255, 128),   # Spring Green
    20: (128, 255, 0)    # Lime
}


In [ ]:
def plot_boxes(image, boxes, class_ids, confidences, class_names):
    for box, class_id, confidence in zip(boxes, class_ids, confidences):
        x1, y1, x2, y2 = map(int, box)
        color = color_dict[int(class_id)]
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        label = f'{class_names[class_id]}: {confidence:.2f}'
        cv2.putText(image, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        
    return image

In [ ]:
# model_path = './<local-path>'
# model = YOLO(model_path)

In [ ]:
# image_index = 484
# test_image = cv2.imread(image_files[image_index])
# test_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
# results = model.predict(test_image)

In [ ]:
# # Get the results
# boxes = results[0].boxes.xyxy.cpu().numpy()
# confidences = results[0].boxes.conf.cpu().numpy()
# class_ids = results[0].boxes.cls.cpu().numpy().astype(int)
# class_names = model.names
# image_with_boxes = plot_boxes(test_image.copy(), boxes, class_ids, confidences, class_names)
# # image_with_boxes_rgb = cv2.cvtColor(image_with_boxes, cv2.COLOR_BGR2RGB)

In [ ]:
# plt.figure(figsize=(20, 20))
# plt.subplot(1,2,1)
# plt.imshow(test_image)

# plt.subplot(1,2,2)
# plt.imshow(image_with_boxes)
# # plt.show()